# First time trying OneMoreEpoch

I've never used this library before — this is what happens when you follow
the README from a completely clean machine: install it with `pip`, import
it, and try the things a new user would actually want to try.

Every output below is real, captured from an actual run of
`onemoreepoch==0.1.2` (as published on TestPyPI) in a brand-new virtual
environment, executed from a directory outside the project's own source
tree. Nothing here is fabricated or guessed.

In [1]:
%pip install --index-url https://test.pypi.org/simple/ \
  --extra-index-url https://pypi.org/simple/ \
  onemoreepoch

Looking in indexes: https://test.pypi.org/simple/, https://pypi.org/simple/
  Using cached onemoreepoch-0.1.2-cp312-cp312-win_amd64.whl.metadata (9.5 kB)
  Using cached numpy-2.5.2-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
Using cached onemoreepoch-0.1.2-cp312-cp312-win_amd64.whl (248 kB)
Using cached numpy-2.5.2-cp312-cp312-win_amd64.whl (12.5 MB)



## Step 1 — import it and see what we got

If this is really the installed package (not some local copy), `__file__`
should point into `site-packages`.

In [2]:
import numpy as np
import onemoreepoch
from onemoreepoch import config, nn
from onemoreepoch.core import Tensor
from onemoreepoch.core.backend import get_backend
from onemoreepoch.optim import Adam, SGD
from onemoreepoch.data import DataLoader, TensorDataset
from onemoreepoch import metrics

print("onemoreepoch version:", onemoreepoch.__version__)
print("installed at:", onemoreepoch.__file__)
print("numpy version:", np.__version__)

onemoreepoch version: 0.1.0
installed at: C:\Users\akash\AppData\Local\Temp\claude\E--OneMoreEpoch\4ac62229-d6b8-4e00-b96d-23771c866c2e\scratchpad\nb2_venv\Lib\site-packages\onemoreepoch\__init__.py
numpy version: 2.5.2


> **Bug found while writing this notebook:** the version above prints
> `0.1.0`, even though the actual installed package is `0.1.2` — a real bug
> (a hardcoded version string that drifted from `pyproject.toml`). It's been
> fixed in the source and will be correct starting with the next published
> release. This output is left exactly as captured, since it's genuinely
> what today's published package does.

## Step 2 — the basics: Tensor, arithmetic, autograd

In [3]:
x = Tensor([1.0, 2.0, 3.0], requires_grad=True)
y = Tensor([4.0, 5.0, 6.0], requires_grad=True)
z = x * y + x
print("x =", x)
print("y =", y)
print("z = x * y + x =", z)

z.sum().backward()
print("dz/dx =", x.grad)
print("dz/dy =", y.grad)

x = Tensor(array([1., 2., 3.]), requires_grad=True)
y = Tensor(array([4., 5., 6.]), requires_grad=True)
z = x * y + x = Tensor(array([ 5., 12., 21.]), requires_grad=True)
dz/dx = [5. 6. 7.]
dz/dy = [1. 2. 3.]


## Step 3 — check the backends

The README mentioned a Rust engine. Let's see if it's actually there.

In [4]:
print("default backend:", get_backend().name)
rust_backend = get_backend("rust")
print("rust backend loaded:", rust_backend.name, "->", type(rust_backend).__name__)

a = Tensor(np.array([1.0, 2.0, 3.0]))
b = Tensor(np.array([10.0, 20.0, 30.0]))
print("numpy backend result:", (a + b).numpy())

default backend: numpy
rust backend loaded: rust -> RustBackend
numpy backend result: [11. 22. 33.]


It loaded straight from the wheel — no Rust toolchain needed on this
machine, because a pre-built wheel was used.

## Step 4 — train something real

A small regression problem (`y = 2*x0 - 3*x1 + 0.5`, plus noise), a 2-8-1
MLP, Adam.

In [5]:
rng = np.random.default_rng(0)
true_w = np.array([[2.0], [-3.0]])
true_b = 0.5
x_data = rng.standard_normal((64, 2))
y_data = x_data @ true_w + true_b + 0.05 * rng.standard_normal((64, 1))

model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=0.05)

losses = []
for epoch in range(200):
    optimizer.zero_grad()
    prediction = model(Tensor(x_data))
    loss = criterion(prediction, Tensor(y_data))
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:3d}  loss {loss.item():.6f}")

print("loss decreased:", losses[0] > losses[-1])

epoch   0  loss 10.284481
epoch  40  loss 0.161621
epoch  80  loss 0.039556
epoch 120  loss 0.004959
epoch 160  loss 0.003243
epoch 199  loss 0.002907
loss decreased: True


## Step 5 — evaluate it

In [6]:
predictions = model(Tensor(x_data)).numpy()
print("MSE:", metrics.mse(predictions, y_data))
print("MAE:", metrics.mae(predictions, y_data))

MSE: 0.0029010004862922555
MAE: 0.043329867138906714


## Step 6 — batching real data with `DataLoader`

In [7]:
dataset = TensorDataset(x_data, y_data)
loader = DataLoader(dataset, batch_size=16, shuffle=True, seed=0)
print("dataset size:", len(dataset), "  batches per epoch:", len(loader))
for batch_x, batch_y in loader:
    print("batch shapes:", batch_x.shape, batch_y.shape)
    break

dataset size: 64   batches per epoch: 4
batch shapes: (16, 2) (16, 1)


## Step 7 — comparing optimizers

In [8]:
for OptCls, kwargs in [(SGD, {"lr": 0.1, "momentum": 0.9}), (Adam, {"lr": 0.05})]:
    m = nn.Linear(2, 1)
    opt = OptCls(m.parameters(), **kwargs)
    for _ in range(50):
        opt.zero_grad()
        pred = m(Tensor(x_data))
        l = criterion(pred, Tensor(y_data))
        l.backward()
        opt.step()
    print(OptCls.__name__, "final loss:", l.item())

SGD final loss: 0.04033305700866083
Adam final loss: 0.23697485350942524


## Step 8 — the rest of `nn`: `Conv2D`, `BatchNorm`, `Dropout`

The regression example only used `Linear`. Checking the other layers exist
and actually work too.

In [9]:
images = Tensor(rng.standard_normal((4, 3, 16, 16)).astype(np.float64))
conv = nn.Conv2D(3, 8, kernel_size=3, stride=2, padding=1)
conv_out = conv(images)
print("Conv2D output shape:", conv_out.shape)

# BatchNorm here normalizes over (N, C), so flatten the spatial dims into features first.
flat = conv_out.reshape(conv_out.shape[0], -1)
bn = nn.BatchNorm(flat.shape[1])
normalized = bn(flat)
print("BatchNorm output shape:", normalized.shape)
print("per-feature mean ~0:", np.allclose(normalized.numpy().mean(axis=0), 0, atol=1e-5))

dropout = nn.Dropout(0.5)
dropout.train()
dropped = dropout(Tensor(np.ones(2000)))
print("Dropout keep fraction (~0.5 expected):", float((dropped.numpy() != 0).mean()))
dropout.eval()
identity = dropout(Tensor(np.ones(5)))
print("Dropout in eval mode is identity:", identity.numpy().tolist())

Conv2D output shape: (4, 8, 8, 8)
BatchNorm output shape: (4, 512)
per-feature mean ~0: True
Dropout keep fraction (~0.5 expected): 0.499
Dropout in eval mode is identity: [1.0, 1.0, 1.0, 1.0, 1.0]


## Step 9 — saving and loading a trained model

`state_dict()` / `load_state_dict()` — the same pattern as PyTorch.

In [10]:
saved_state = model.state_dict()
print("saved parameter names:", list(saved_state.keys()))

fresh_model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))
before = fresh_model(Tensor(x_data[:1])).item()
fresh_model.load_state_dict(saved_state)
after = fresh_model(Tensor(x_data[:1])).item()
print("prediction before loading weights:", before)
print("prediction after loading weights:", after)
print("matches trained model:", np.isclose(after, model(Tensor(x_data[:1])).item()))

saved parameter names: ['0.weight', '0.bias', '2.weight', '2.bias']
prediction before loading weights: -0.15743278687131618
prediction after loading weights: 1.1406319780589986
matches trained model: True


## Step 10 — the personality system

Same error, same underlying exception, three different voices. The
technical message never disappears — the personality layer only adds to it.

### `classic` (the default)

In [11]:
config.set_message_mode("classic")
try:
    Tensor.randn(64, 128) @ Tensor.randn(32, 10)
except Exception as e:
    print(str(e))

Matrix multiplication shape mismatch.
Left matrix : (64, 128)
Right matrix: (32, 10)
Inner dimensions must agree (left columns == right rows).

Shapes didn't line up. Check your dimensions before the next forward pass.


### `hindi`

**This answers a question I had:** do I need to call `set_message_mode`
before *every* operation? No — it's a one-time switch per process. Set it
once, and it stays in effect for everything after, in this same run:

In [12]:
config.set_message_mode("hindi")
print("mode is now:", config.get_message_mode())
try:
    Tensor.randn(3, 2) + Tensor.randn(4, 5)
except Exception as e:
    print(str(e))
print("--- and it stays hindi for every later call in THIS process, no need to set it again ---")
try:
    Tensor.randn(5, 5) @ Tensor.randn(3, 3)
except Exception as e:
    print(str(e))

mode is now: hindi
🚨 Arre Add karne chale the, par shapes ((3, 2), (4, 5)) ka jodi nahi bana.
Broadcasting ke rishte mein har dimension barabar ya 1 honi chahiye.

🎬 Dono shapes ka scene match nahi kar raha, poora climax alag reel se hai.
--- and it stays hindi for every later call in THIS process, no need to set it again ---
🚨 Arre bhai!
Shape mismatch.
Left Matrix : (5, 5)
Right Matrix: (3, 3)
Ye shaadi nahi ho sakti.

🎬 Wrong train pakad li tune, station hi alag hai - shapes match nahi.


### `roast`

In [13]:
config.set_message_mode("roast")
try:
    Tensor.randn(64, 128) @ Tensor.randn(32, 10)
except Exception as e:
    print(str(e))

config.set_message_mode("classic")

🔥 You tried to matmul (64, 128) with (32, 10).
Those inner dimensions don't match, and honestly, neither do you and linear algebra right now.

🔥 Your tensors just filed for divorce over irreconcilable shapes.


## Step 11 — personality during training (`TrainingLogger`)

In [14]:
config.set_message_mode("hindi")
from onemoreepoch.utils import TrainingLogger

logger = TrainingLogger()
tiny_model = nn.Linear(2, 1)
tiny_opt = SGD(tiny_model.parameters(), lr=0.1)
for epoch in range(5):
    tiny_opt.zero_grad()
    pred = tiny_model(Tensor(x_data))
    loss = criterion(pred, Tensor(y_data))
    loss.backward()
    tiny_opt.step()
    logger.log_epoch(epoch, loss.item())
logger.finish()

config.set_message_mode("classic")

Epoch    0 | Loss: 14.066876  Model: "Aaj kuch toofani karte hain."
Epoch    1 | Loss: 9.857537  Model: "Gradient flow full speed pe hai."
Epoch    2 | Loss: 6.941892  Model: "Loss neeche, confidence upar."
Epoch    3 | Loss: 4.909814  Model: "Ek aur epoch, ek aur kamaal."
Epoch    4 | Loss: 3.485700  Model: "Weights set, scene set."
🎉 Training khatam: 5 epochs, best loss 3.485700.
Ek aur epoch? Naam hi OneMoreEpoch hai.


## Step 12 — making a personality mode *permanent* (no code needed)

`config.set_message_mode()` only lasts for the current process — a new
script run or a new notebook kernel starts back at `classic`. For "set it
once, forever, without touching code," there's an environment variable
(`ONEMOREEPOCH_MESSAGES`) that `onemoreepoch` reads automatically the moment
it's imported.

Proving it actually works, by launching two completely separate Python
processes — one with the variable set, one without — and checking the mode
each one starts in, before either one calls `set_message_mode` at all:

In [15]:
import subprocess, sys, os

proof_script = (
    "import onemoreepoch.config as config;"
    "print('starting mode:', config.get_message_mode())"
)

env_with_hindi = dict(os.environ)
env_with_hindi["ONEMOREEPOCH_MESSAGES"] = "hindi"
with_var = subprocess.run(
    [sys.executable, "-c", proof_script], env=env_with_hindi, capture_output=True, text=True
)
print("with ONEMOREEPOCH_MESSAGES=hindi set:", with_var.stdout.strip())

env_without = dict(os.environ)
env_without.pop("ONEMOREEPOCH_MESSAGES", None)
without_var = subprocess.run(
    [sys.executable, "-c", proof_script], env=env_without, capture_output=True, text=True
)
print("with no env var set:               ", without_var.stdout.strip())

mode at fresh import, with ONEMOREEPOCH_MESSAGES=hindi set: hindi
mode at fresh import, with no env var set: classic


Neither of those subprocesses ever called `set_message_mode()` — the
env var alone decided the starting mode. To make hindi permanent on your
own machine:

- **Windows (PowerShell), current session only:**
  ```powershell
  $env:ONEMOREEPOCH_MESSAGES = "hindi"
  ```
- **Windows, permanently for your user account** (new terminals/notebooks
  pick it up automatically from then on):
  ```powershell
  [System.Environment]::SetEnvironmentVariable("ONEMOREEPOCH_MESSAGES", "hindi", "User")
  ```
  (Restart the terminal/VS Code afterward — existing sessions won't see it
  until then.)
- **macOS/Linux, permanently:** add `export ONEMOREEPOCH_MESSAGES=hindi` to
  `~/.bashrc` / `~/.zshrc`.

`EDUCATIONAL_MODE=1` also works as a shorthand for hindi, if you prefer that
name.

## Done

Everything above ran against the real published package, in a clean
environment, with no editable install and no local source on the path.

In [16]:
print("All checks completed successfully.")

All checks completed successfully.
